In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
data = pd.read_csv('../data/processed_shots.csv')
data.head()

In [ ]:
# Outcome classes: 0 = miss, 1 = behind, 2 = goal. Expected points = P(behind) * 1 + P(goal) * 6
CLASS_POINTS = np.array([0, 1, 6])
y = data['Score'].map({0: 0, 1: 1, 6: 2})

In [ ]:
# Soccer-xG style polynomial terms
data['dist_sq'] = data['dist_to_goal'] ** 2
data['angle_x_dist'] = data['angle_to_goal'] * data['dist_to_goal']

# Angle subtended by the behind posts (6.4m beyond each goal post, so at +/-9.6m)
x, lateral = data['x'], data['y']
dist_l = np.sqrt(x**2 + (9.6 - lateral)**2)
dist_r = np.sqrt(x**2 + (-9.6 - lateral)**2)
data['angle_behind'] = np.degrees(np.arccos(np.clip((dist_l**2 + dist_r**2 - 19.2**2) / (2 * dist_l * dist_r), -1, 1)))
data['angle_outer'] = data['angle_behind'] - data['angle_to_goal']
data['dist_behind_near'] = np.minimum(dist_l, dist_r)

base = ['dist_to_goal', 'angle_to_goal', 'set_shot_flag']
poly = base + ['dist_sq', 'angle_x_dist']
behind = poly + ['angle_behind', 'angle_outer', 'dist_behind_near']
feature_sets = {'base': base, 'poly': poly, 'poly + behind posts': behind}

In [ ]:
# 5-fold CV comparison of the feature sets
def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

def score(proba, y_true, points):
    expected = proba @ CLASS_POINTS
    return log_loss(y_true, proba), np.sqrt(np.mean((points - expected) ** 2))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, cols in feature_sets.items():
    proba = cross_val_predict(make_model(), data[cols], y, cv=kf, method='predict_proba')
    ll, rmse = score(proba, y, data['Score'])
    rows.append({'features': name, 'log_loss': ll, 'expected_points_rmse': rmse})
pd.DataFrame(rows).round(4)

In [ ]:
# Untuned model on a train/test split (hand-built polynomial terms)
features = poly
X_train, X_test, y_train, y_test = train_test_split(data[features], y, test_size=0.2, random_state=42)

model = make_model()
model.fit(X_train, y_train)

test_proba = model.predict_proba(X_test)
ll, rmse = score(test_proba, y_test, data.loc[X_test.index, 'Score'])
print(f"Untuned test log loss: {ll:.4f}")
print(f"Untuned test expected-points RMSE: {rmse:.4f}")

In [ ]:
# Tune the polynomial degree and regularization strength (C) with 5-fold CV on the training set.
# PolynomialFeatures builds the squared and interaction terms itself, so the input is the raw columns.
raw_sets = {
    'raw': base,
    'raw + behind posts': base + ['angle_behind', 'angle_outer', 'dist_behind_near'],
}
pipe = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000)),
])
param_grid = {
    'poly__degree': [1, 2, 3, 4],
    'clf__C': np.logspace(-3, 2, 8),
}

searches = {}
for name, cols in raw_sets.items():
    gs = GridSearchCV(pipe, param_grid, scoring='neg_log_loss', cv=KFold(n_splits=5, shuffle=True, random_state=42), n_jobs=-1)
    gs.fit(data.loc[X_train.index, cols], y_train)
    searches[name] = gs
    print(f"{name}: CV log loss {-gs.best_score_:.4f}  {gs.best_params_}")

best_name = max(searches, key=lambda n: searches[n].best_score_)
search = searches[best_name]
best_cols = raw_sets[best_name]
print(f"\nBest feature set: {best_name}")

In [ ]:
# Evaluate the tuned model on the held-out test set
best_model = search.best_estimator_
tuned_proba = best_model.predict_proba(data.loc[X_test.index, best_cols])
tuned_ll, tuned_rmse = score(tuned_proba, y_test, data.loc[X_test.index, 'Score'])
print(f"Tuned test log loss:   {tuned_ll:.4f}  (untuned {ll:.4f})")
print(f"Tuned test expected-points RMSE: {tuned_rmse:.4f}  (untuned {rmse:.4f})")

In [ ]:
# Coefficients on standardized features (one row per class)
names = best_model[:-1].get_feature_names_out(best_cols)
coefs = pd.DataFrame(best_model[-1].coef_, index=['miss', 'behind', 'goal'], columns=names)
coefs.round(3)

In [ ]:
probas = best_model.predict_proba(data[best_cols])
data['p_miss'], data['p_behind'], data['p_goal'] = probas.T
data['mlr_xscore'] = probas @ CLASS_POINTS
data[['dist_to_goal', 'angle_to_goal', 'Score', 'xScore', 'p_miss', 'p_behind', 'p_goal', 'mlr_xscore']].head()

In [ ]:
# Baseline: existing xScore vs actual Score
rmse = np.sqrt(np.mean((data['Score'] - data['xScore']) ** 2))
print(f"xScore RMSE: {rmse:.4f}")